# Module 3 — Tìm thời điểm có điều kiện không khí tương đồng

Bản hoàn chỉnh dùng đủ 12 trạm Beijing. Hai phạm vi: cùng trạm và toàn kho. Hai khoảng cách: Euclidean và Manhattan. Notebook dùng lại pipeline trong `train.py`; mô hình ứng dụng là `knn.joblib`, không tạo thêm một bản `notebook_knn.joblib` riêng.

In [ ]:
from pathlib import Path
import sys, json
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "Module 3"]
ROOT = next((candidate for candidate in candidates if (candidate / "module3" / "train.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Mở notebook trong repo ML hoặc thư mục Module 3.")
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, Markdown
from module3.data import FEATURES, read_data, prepare_data, split_by_time
from module3.similarity import SimilaritySearch, find_similar_periods
from module3.prepare import get_data_source, prepare
ARTIFACTS = ROOT / "artifacts/module3"
SOURCE = get_data_source()


## 1. Nguồn và chất lượng dữ liệu
UCI Beijing Multi-Site Air Quality, DOI 10.24432/C5RK5G, CC BY 4.0. Ô nhiễm từ Beijing Municipal Environmental Monitoring Center; khí tượng từ China Meteorological Administration. Dữ liệu 2013–2017, giờ Bắc Kinh UTC+8. Không mã hóa mã trạm hoặc timestamp thành biến khoảng cách.

In [ ]:
raw = read_data(SOURCE)
df, quality = prepare_data(raw)  # mặc định tất cả các trạm
print(f"Dữ liệu gốc: {len(raw):,} dòng; số trạm: {raw.station.nunique()}")
print(f"Sử dụng: {len(df):,}; loại do thiếu/không hợp lệ: {quality['rows_dropped_missing']:,}")
display(pd.DataFrame(quality["by_station"]))

## 2. Chia thời gian và tránh thông tin tương lai
Train trước 01/11/2015; validation trước 01/07/2016; phần còn lại là test. Chỉ train tham gia scaler và kho lịch sử. Thiếu một trong sáu biến thì loại dòng, không nội suy. Mỗi trạm phải có dữ liệu trong cả ba tập.

In [ ]:
train, valid, test = split_by_time(df)
display(pd.DataFrame({"split":["train","validation","test"], "rows":[len(train),len(valid),len(test)], "stations":[part.station.nunique() for part in [train,valid,test]], "start":[part.datetime.min() for part in [train,valid,test]], "end":[part.datetime.max() for part in [train,valid,test]]}))

## 3. Xây bộ tìm kiếm và khảo sát
Đặt REBUILD=True để tạo lại. Mặc định tự chuẩn bị khi thiếu artifact hoặc dữ liệu/mã/thư viện thay đổi; các lần sau dùng lại. Không cần gửi joblib qua Git. Dữ liệu ưu tiên 12 CSV PRSA tại data/ của repo; nếu thiếu sẽ dùng ZIP cache hoặc tải UCI.

StandardScaler chỉ học từ train. Validation khảo sát K=3/5/10, Euclidean/Manhattan và hai phạm vi. Đối chứng ngẫu nhiên lấy cùng phạm vi. Không so trực tiếp giá trị khoảng cách của hai metric để chọn mô hình.


In [ ]:
REBUILD = False
OUTPUT_DIR = ARTIFACTS
metadata = prepare(data=SOURCE, output=OUTPUT_DIR, force=REBUILD, queries=120)
metrics = pd.read_csv(OUTPUT_DIR / "validation_metrics.csv")
display(metrics)


In [ ]:
display(Image(filename=str(OUTPUT_DIR / "station_coverage.png")))
display(Image(filename=str(OUTPUT_DIR / "validation_comparison.png")))

## 4. Một truy vấn, hai phạm vi tìm kiếm
Ví dụ lấy từ validation sau kho train. `station=None` theo trạm của query; `station="ALL"` tìm toàn bộ 12 trạm. Đầu ra giữ nguyên đơn vị gốc.

In [ ]:
searcher = SimilaritySearch.load(OUTPUT_DIR / "knn.joblib")
query = valid.iloc[0].to_dict()
display(pd.DataFrame([query])[["datetime","station",*FEATURES]])
same_station = find_similar_periods(query, k=5, searcher=searcher)
all_stations = find_similar_periods(query, k=5, searcher=searcher, station="ALL")
print("Cùng trạm")
display(same_station)
print("Toàn bộ trạm")
display(all_stations)

In [ ]:
gaps = all_stations[["rank","station","datetime"]].copy()
for feature in FEATURES:
    gaps["delta_" + feature] = all_stations[feature] - float(query[feature])
display(gaps)

## 5. Đánh giá cuối trên test
Cấu hình mặc định Euclidean, K=5 được đặt trước. Báo cáo test cho hai phạm vi, có chi tiết theo trạm. Không chỉnh cấu hình từ kết quả test. Khoảng cách và chênh lệch đặc trưng là chỉ báo truy xuất, không phải độ chính xác dự báo.

In [ ]:
display(pd.read_csv(OUTPUT_DIR / "test_metrics.csv"))
display(pd.read_csv(OUTPUT_DIR / "test_metrics_by_station.csv"))
display(Image(filename=str(OUTPUT_DIR / "example_comparison.png")))

## 6. Diễn giải và bàn giao
- PM2.5 nằm trong đầu vào: kết quả gần về PM2.5 không phải đánh giá độc lập.
- KNN luôn trả đủ K nếu có đủ ứng viên, dù chúng có thể ở xa. Không diễn giải distance thành phần trăm.
- Dữ liệu thiếu bị loại có thể làm lệch mẫu. Các giờ liền nhau có thể cùng xuất hiện.
- Đây là tìm quan sát theo giờ, chưa tìm cả chuỗi nhiều giờ hoặc dự báo.
- Cùng scaler toàn kho giúp dùng nhất quán khi tìm cùng trạm và xuyên trạm; chưa học trọng số tầm quan trọng các biến.

Bàn giao cả mã nguồn, README và artifact, hoặc chạy `python -m module3.train` để tái tạo. Ứng dụng chạy bằng `python -m streamlit run module3/app.py`.

In [ ]:
display(Markdown((OUTPUT_DIR / "REPORT_MODULE3.md").read_text(encoding="utf-8")))